# Thai synthetic speech on Colab — VoxCPM

Generates Thai TTS clips for the ASR adapter experiment: **16 kHz mono**, one clip per
text, spoken by 150 CC0 Common Voice speakers. Sharded to Drive and **resumable** — you
will be disconnected, and that is fine.

**Run the cells in order. Do the smoke test before the long run.**

Full reasoning: `colab/README.md` and the docstring of `gen_colab.py`.


## 1. Check the GPU

Pick **L4** if you have the choice. VoxCPM's DiT runs on a tiny `(2,64,4)` tensor and is
launch-bound, not compute-bound — that is why TensorRT only *tied* `torch.compile` on it.
An A100 buys almost nothing here and burns ~5x the compute units.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv


## 2. Install

`voxcpm` is a normal pip package, so there is no repo to clone. Pinned to the version the
local box runs — an unpinned install is a different model on a different day.


In [ ]:
!pip install -q voxcpm==2.0.3 librosa
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())


## 3. Mount Drive

Drive is the **only** thing that survives a disconnect. The VM's own disk does not.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 4. Put the inputs in place

You need three things on Drive, once:

| what | size | where from |
|---|---|---|
| `gen_colab.py` | 15 KB | this repo, `colab/` |
| `refs/` (150 wavs + `refs.json`) | 82 MB | this repo, `smartturn/refs/` |
| a text bank `.json` | small | see README — **this is the open decision** |

Upload `colab-bundle.zip` (built by the README's one-liner) to
`MyDrive/thai-synth/` and run the cell below to unpack it.


In [ ]:
WORK = '/content/drive/MyDrive/thai-synth'

import os, zipfile, pathlib
bundle = f'{WORK}/colab-bundle.zip'
if os.path.exists(bundle) and not os.path.isdir(f'{WORK}/refs'):
    with zipfile.ZipFile(bundle) as z:
        z.extractall(WORK)
    print('unpacked')

for p in ['gen_colab.py', 'refs/refs.json']:
    print(('OK   ' if os.path.exists(f'{WORK}/{p}') else 'MISSING '), p)
print('text banks:', [f.name for f in pathlib.Path(WORK).glob('*.json')])


## 5. Smoke test — 3 clips, then LISTEN

This is the only check that VoxCPM actually works on the GPU Colab handed you. The local
box is a Blackwell card with `torch.compile` and CUDA graphs; Colab is not, and a model
card is not evidence.

The clips land as loose wavs in `thai-synth/smoke/`. **Open them from Drive and listen.**
A waveform metric cannot tell you the voice is right — this repo has ~5 confident wrong
conclusions on record from trusting one.

First load is slow (model download, ~2 GB).


In [ ]:
TEXTS = f'{WORK}/texts_v3.json'   # <-- the bank to speak

!cd {WORK} && python gen_colab.py \
    --texts {TEXTS} --refs {WORK}/refs --out {WORK} --smoke 3


## 6. The run

Resumable: it reads the manifests already on Drive and skips what is done, so **just
re-run this cell after a disconnect.** Nothing is lost but the shard in flight, and that is
regenerated bit-identically because every clip's seed comes from its own id, never from its
position in the run.

`--limit` caps one session so you can stop on your own terms rather than Colab's. Raise
`--shard-clips` only if Drive writes feel slow; lower it if you keep losing shards.

Rough cost, from the local box's rate (RTF ~0.29): **~1 hour of audio per 17 min of
generation**, and audio is ~115 MB/hour. The script prints its own measured RTF — use that,
not this estimate.


In [ ]:
!cd {WORK} && python gen_colab.py \
    --texts {TEXTS} --refs {WORK}/refs --out {WORK} \
    --shard-clips 500 --limit 2000


## 7. What you have


In [ ]:
import json, pathlib, collections
rows = []
for m in sorted(pathlib.Path(f'{WORK}/manifests').glob('*.jsonl')):
    rows += [json.loads(l) for l in m.open(encoding='utf-8') if l.strip()]
secs = sum(r['dur_s'] for r in rows)
voices = len({r['client_id'] for r in rows})
print(f'{len(rows)} clips | {secs/3600:.2f} h of audio | {voices} voices')
print('shards:', len(list(pathlib.Path(f'{WORK}/shards').glob('*.tar'))))
print('by class:', dict(collections.Counter(r.get('cls') for r in rows)))


---
### Bringing it home

Download the `shards/` tars and the `manifests/`, then untar into one `clips/` directory.
Keep the manifests — they carry the text, voice and seed for every clip, which is what
makes the corpus auditable rather than just a pile of wavs.
